# 03 — Published Results and Reproducibility Receipts

This notebook demonstrates how to browse published results and inspect the
reproducibility receipt fields that make every published result reconstructable.

## Architecture note (§5 Reproducibility receipt)

For each published result the system records:
- `manifest_hash` — hash of the ingest manifest (links back to declared file set)
- `raw_file_hashes_json` — per-file sha256 hashes of the raw data
- `parameter_version_ids_json` — sample stack, contact geometry, processing parameter versions used
- `parser_version` — the exact parser code version used
- `published_by` — who approved publication

Together these form an audit trail from published output back to raw bytes.

**Access:** read-only via `labdata.catalog` (`v_published_results`, `v_run_files`).

In [ ]:
import json
import pandas as pd

import labdata.catalog as catalog

pd.set_option('display.max_colwidth', 60)
pd.set_option('display.max_columns', 20)

## 1. List all published results

`catalog.published_results()` queries `v_published_results`.

In [ ]:
df_pub = catalog.published_results()
print(f"Published results: {len(df_pub)}")
df_pub.head()

In [ ]:
if df_pub.empty:
    raise RuntimeError(
        "No published results found. Seed demo data first:\n"
        "  from labdata.seed import seed_demo; seed_demo()"
    )

# Pick the first published result
pub = df_pub.iloc[0]
pub_result_id = pub['id']
run_id = pub['run_id']
print(f"Inspecting published_result_id: {pub_result_id}")
print(f"For run_id:                     {run_id}")

## 2. Inspect the reproducibility receipt fields

These fields are what makes the result reconstructable from scratch.

In [ ]:
print("=== Reproducibility Receipt ===")
print()
print(f"published_result_id:         {pub_result_id}")
print(f"run_id:                      {run_id}")
print()

# manifest_hash — links back to the declared file set at ingest time
print(f"manifest_hash:               {pub.get('manifest_hash', 'N/A')}")
print()

# raw_file_hashes_json — per-file sha256 of raw data
raw_hashes_raw = pub.get('raw_file_hashes_json')
if raw_hashes_raw:
    if isinstance(raw_hashes_raw, str):
        raw_hashes_raw = json.loads(raw_hashes_raw)
    # raw_file_hashes_json is a list of {name, sha256} (Go publisher shape); normalize to a dict.
    raw_hashes = dict(raw_hashes_raw) if isinstance(raw_hashes_raw, dict) else {fh['name']: fh['sha256'] for fh in raw_hashes_raw}
    print("raw_file_hashes_json:")
    for fname, sha in raw_hashes.items():
        print(f"  {fname}: {sha}")
else:
    print("raw_file_hashes_json:        (none)")
print()

# parameter_version_ids_json — which parameter versions were used for this result
param_vers_raw = pub.get('parameter_version_ids_json')
if param_vers_raw:
    param_vers = param_vers_raw if isinstance(param_vers_raw, dict) else json.loads(param_vers_raw)
    print("parameter_version_ids_json:")
    for k, v in param_vers.items():
        print(f"  {k}: {v}")
else:
    print("parameter_version_ids_json:  (none)")
print()

# parser_version — exact version string of the parser code
print(f"parser_version:              {pub.get('parser_version', 'N/A')}")

# processing_git_sha — Git SHA of the processing code
print(f"processing_git_sha:          {pub.get('processing_git_sha', 'N/A')}")

# processing_dirty_tree — True if working tree was dirty at processing time
print(f"processing_dirty_tree:       {pub.get('processing_dirty_tree', 'N/A')}")

# published_by — who approved this result
print(f"published_by:                {pub.get('published_by', 'N/A')}")
print(f"published_at:                {pub.get('published_at', 'N/A')}")

## 3. Run files with sha256

`catalog.run_files(run_id)` returns each file in the raw archive with its
recorded sha256 hash (from `v_run_files`). These are the hashes verified
at promotion time and recorded in the reproducibility receipt.

In [ ]:
df_files = catalog.run_files(run_id)
print(f"Run files for {run_id[:12]}...:")
df_files[['name', 'bytes', 'sha256', 'created_at']]

## 4. Cross-reference: do raw_file_hashes_json match run_files?

The `raw_file_hashes_json` in the published result should match the sha256
values in `v_run_files`. This cross-check confirms consistency between the
receipt and the catalog.

(For a full re-read-from-disk check, see `shared/reproducibility_demo.ipynb`.)

In [ ]:
if raw_hashes_raw and not df_files.empty:
    if isinstance(raw_hashes_raw, str):
        raw_hashes_raw = json.loads(raw_hashes_raw)
    # raw_file_hashes_json is a list of {name, sha256} (Go publisher shape); normalize to a dict.
    raw_hashes = dict(raw_hashes_raw) if isinstance(raw_hashes_raw, dict) else {fh['name']: fh['sha256'] for fh in raw_hashes_raw}
    mismatches = []
    for _, row in df_files.iterrows():
        fname = row['name']
        expected_sha = raw_hashes.get(fname)
        actual_sha = row['sha256']
        if expected_sha and expected_sha != actual_sha:
            mismatches.append((fname, expected_sha, actual_sha))

    if mismatches:
        print("MISMATCHES detected:")
        for fname, exp, act in mismatches:
            print(f"  {fname}: receipt={exp} vs catalog={act}")
    else:
        print("All raw file hashes in receipt match catalog records.")
else:
    print("Skipping cross-check (no raw_file_hashes or no run_files).")